Import libraries

In [ ]:
import os
import pandas as pd
import tensorflow as tf

In [ ]:
import matplotlib
from matplotlib import pyplot as plt
COLORS = matplotlib.rcParams['axes.prop_cycle'].by_key()['color'] * 30

Inputs

In [ ]:
# System
atom = "O"
molecule = "O2"
nb_bins = 45
# Paths
path_to_model = "./../saving/full_me/SavedModel/"
path_to_data = "./../../../datagen/data/test/"
path_to_figs = "./no_ref/"

Load model

In [ ]:
model = tf.keras.models.load_model(path_to_model)

Get solution

In [ ]:
lbl_mol = [f"n_{molecule}_{i+1}" for i in range(nb_bins)]

def get_data(filename):
  sol = pd.read_csv(filename)
  # Extract inputs
  na_0, T_0 = sol[[f"n_{atom}_0"]].to_numpy(), sol[["T_0"]].to_numpy()
  nmi_0 = sol[[k+"_0" for k in lbl_mol]].to_numpy()
  t = sol[["t"]].to_numpy()
  # Extract exact solution
  na, T = sol[[f"n_{atom}"]].to_numpy(), sol[["T"]].to_numpy()
  nmi = sol[lbl_mol].to_numpy()
  y_true = [na, nmi, T]
  # Get predicted solution
  inputs = [na_0, nmi_0, T_0, t]
  y_pred = [yi.numpy() for yi in model(inputs)]
  return t, y_true, y_pred

Plot solution

In [ ]:
def plot_data(
  x=None,
  y=None,
  xref=None,
  yref=None,
  labels=[r'$t$ [s]', r'$X$'],
  scales=['log', 'log'],
  names=None,
  figname=None
):
  # Initialize figures
  fig = plt.figure()
  ax = fig.add_subplot()
  # x axis
  ax.set_xlabel(labels[0])
  ax.set_xscale(scales[0])
  # y axis
  ax.set_ylabel(labels[1])
  ax.set_yscale(scales[1])
  # Plotting
  n, d = y.shape if (x is not None) else yref.shape
  true_style = dict(
    lw=1
  )
  pred_style = dict(
    marker='^',
    fillstyle='none',
    markersize=5,
    markevery=n//20,
    ls='--',
    lw=1
  )
  for i in range(d):
    if (x is not None):
      ax.plot(x, y[:,i], c=COLORS[i], **pred_style)
    if (xref is not None):
      ax.plot(xref, yref[:,i], c=COLORS[i], **true_style)
    if (names is not None):
      ax.plot([], [], c=COLORS[i], label=names[i])
  # Tight layout
  plt.tight_layout()
  if (names is not None):
    plt.legend()
  if (figname is not None):
    fig.savefig(figname)
  plt.close()

In [ ]:
for (i, case) in enumerate(os.listdir(path_to_data)):
  # Set path for saving
  filename = path_to_data + "/" + case
  path_to_figsi = path_to_figs + "/" + case.split(".")[0]
  os.makedirs(path_to_figsi, exist_ok=True)
  # Get data
  t, y_true, y_pred = get_data(filename)
  # Plot
  plot_data(
    x=t,
    y=y_pred[0],
    xref=t,
    yref=y_true[0],
    labels=[r'$t$ [s]', r'$n$ [m$^{-3}$]'],
    scales=['log', 'linear'],
    figname=path_to_figsi + f"/mass.png"
  )
  plot_data(
    x=t,
    y=y_pred[2],
    xref=t,
    yref=y_true[2],
    labels=[r'$t$ [s]', r'$T$ [K]'],
    scales=['log', 'linear'],
    figname=path_to_figsi + f"/temp.png"
  )
  plot_data(
    x=t,
    y=y_pred[1],
    xref=t,
    yref=y_true[1],
    labels=[r'$t$ [s]', r'$n$ [m$^{-3}$]'],
    scales=['log', 'linear'],
    figname=path_to_figsi + f"/dist_lin.png"
  )
  plot_data(
    x=t,
    y=y_pred[1],
    xref=t,
    yref=y_true[1],
    labels=[r'$t$ [s]', r'$n$ [m$^{-3}$]'],
    scales=['log', 'log'],
    figname=path_to_figsi + f"/dist_log.png"
  )